In [0]:
%run "../00.common/01.environment-config.py"

In [0]:
# Databricks notebook source

# MAGIC %md
# MAGIC # Build Drivers Dimension
# MAGIC
# MAGIC 1. Read Silver `drivers` data
# MAGIC 2. Read Gold `ref_nationality_region` data
# MAGIC 3. Join using `nationality`
# MAGIC 4. Rename `full_name` to `driver_name`
# MAGIC 5. Select the required business columns
# MAGIC 6. Write the transformed data to Gold `dim_drivers`

# COMMAND ----------

# MAGIC %run "../00.common/01.environment-config.py"

# COMMAND ----------

from pyspark.sql import functions as F

# COMMAND ----------

# ============================================================
# SOURCE AND TARGET PATHS
# ============================================================

silver_path = f"{s3_root_path}/Silver/Nisarga/drivers"

ref_path = f"{s3_root_path}/Gold/Nisarga/ref_nationality_region"

gold_path = f"{s3_root_path}/Gold/Nisarga/dim_drivers"

print("Silver Path    :", silver_path)
print("Reference Path :", ref_path)
print("Gold Path      :", gold_path)

# COMMAND ----------

# ============================================================
# STEP 1 - READ SOURCE DATA
# ============================================================

drivers_df = (
    spark.read
    .format("delta")
    .load(silver_path)
)

ref_nationality_region_df = (
    spark.read
    .format("delta")
    .load(ref_path)
)

display(drivers_df)
display(ref_nationality_region_df)

# COMMAND ----------

# ============================================================
# STEP 2 - JOIN DRIVERS WITH NATIONALITY REGION
# ============================================================

dim_drivers_df = (
    drivers_df
    .join(
        ref_nationality_region_df,
        drivers_df.nationality == ref_nationality_region_df.nationality,
        "left"
    )
    .select(
        drivers_df.driver_id,
        drivers_df.full_name.alias("driver_name"),
        drivers_df.date_of_birth,
        drivers_df.nationality,
        ref_nationality_region_df.region.alias("nationality_region")
    )
)

display(dim_drivers_df)

# COMMAND ----------

# ============================================================
# STEP 3 - REMOVE DUPLICATES
# ============================================================

dim_drivers_df = (
    dim_drivers_df
    .dropDuplicates(["driver_id"])
)

display(dim_drivers_df)

# COMMAND ----------

# ============================================================
# STEP 4 - WRITE TO GOLD
# ============================================================

(
    dim_drivers_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(gold_path)
)

print("Gold Drivers data successfully written to:")
print(gold_path)

# COMMAND ----------

# ============================================================
# STEP 5 - VALIDATE GOLD DATA
# ============================================================

gold_drivers_df = (
    spark.read
    .format("delta")
    .load(gold_path)
)

display(gold_drivers_df)

print("Silver Drivers Records :", drivers_df.count())
print("Gold Drivers Records   :", gold_drivers_df.count())

# COMMAND ----------

# ============================================================
# STEP 6 - CHECK NATIONALITY REGION MAPPING
# ============================================================

display(
    gold_drivers_df
    .groupBy("nationality_region")
    .count()
    .orderBy("nationality_region")
)